# ROCF VLM · 복합도형 그리기 과정으로 아밀로이드 베타 예측하기 (합성 데이터 예제)

이 노트북은 **합성 ROCF 특권 정보 활용 학습(Learning Using Privileged Information) 코호트**(`N=400`, `CU + MCI`)를 탐색하고, 정적 그림만 사용하는 기준선이 왜 `MCI` 교란과 `CU` 천장효과에 속는지 직접 확인합니다.

- **학습(`train`, 280명)**: `train_privileged.csv`의 합성 뇌지표(`hippocampal_md`, `precuneus_md`, `hippocampal_vol`, `amyloid_centiloid` 등)·임상 변수·36점 채점표를 모두 활용할 수 있습니다.
- **평가(`test`, 120명)**: 오직 **ROCF 정적 그림(`images/*.png`)과 5 Hz 그리는 과정**(`strokes/*.csv`)만 입력으로 주어집니다.

먼저 터미널에서 합성 데이터를 만들어 두세요:
```bash
python generate_synthetic_rocf.py
```
`data/ground_truth.json`에 블라인드 `test` 정답이 들어 있습니다. **마지막 단계(5절) 전까지 열어 보지 마세요.**


In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import rocf_synth
from starter_pipeline import run_pipeline as run_static_starter

DATA_DIR = Path("data")
part = pd.read_csv(DATA_DIR / "participants.csv")
priv = pd.read_csv(DATA_DIR / "train_privileged.csv")

print("participants.csv 열:", list(part.columns), "| 행 수:", len(part))
print("split 분포:\n", part["split"].value_counts().to_dict())
print("\ntrain_privileged.csv 열:", list(priv.columns))
print("test 참가자가 train_privileged에 포함되었는가?",
      len(set(part.loc[part["split"] == "test", "subject_id"]) & set(priv["subject_id"])) > 0)
priv.head(4)


## 1. 진단군 교란(`CU + MCI`), `CU` 천장효과, 학습 전용 뇌지표(`hippocampal_md`) 탐색

`train_privileged.csv`에서 인지정상(`CU`)과 경도인지장애(`MCI`)의 36점 채점 분포, 아밀로이드 양성률, 그리고 해마 미세구조 지표(`hippocampal_md`)와 해마 부피(`hippocampal_vol`)가 어떻게 다른지 확인합니다.


In [ ]:
summary_tbl = priv.groupby("diagnosis").agg(
    n=("subject_id", "count"),
    ab_pos_rate=("amyloid_pos", "mean"),
    copy_mean=("copy_score_36", "mean"),
    recall_mean=("recall_score_36", "mean"),
    hipp_md_mean=("hippocampal_md", "mean"),
    hipp_vol_mean=("hippocampal_vol", "mean"),
).round(3)
print(summary_tbl)

print("\nCU 내부 아밀로이드 상태별 평균 비교:")
print(priv[priv["diagnosis"] == "CU"].groupby("amyloid_pos")[
    ["copy_score_36", "recall_score_36", "amyloid_centiloid", "hippocampal_md", "precuneus_md", "hippocampal_vol"]
].mean().round(4))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for dx, col in [("CU", "#2a6fdb"), ("MCI", "#d95f02")]:
    sub = priv[priv["diagnosis"] == dx]
    axes[0].hist(sub["copy_score_36"], bins=16, alpha=0.65, label=dx, color=col)
    axes[1].hist(sub["recall_score_36"], bins=16, alpha=0.65, label=dx, color=col)
axes[0].set_title("ROCF Copy Score (36-pt Ceiling in CU)")
axes[0].set_xlabel("Copy Score (/36)"); axes[0].legend()
axes[1].set_title("ROCF Delayed Recall Score")
axes[1].set_xlabel("Recall Score (/36)"); axes[1].legend()

cu = priv[priv["diagnosis"] == "CU"]
for ab, lbl, col in [(0, "CU Ab-", "#2b8cbe"), (1, "CU Ab+", "#e34a33")]:
    s = cu[cu["amyloid_pos"] == ab]
    axes[2].scatter(s["hippocampal_vol"], s["hippocampal_md"], alpha=0.7, s=26, label=lbl, color=col)
axes[2].set_title("Train Privileged Brain Markers (CU Only)")
axes[2].set_xlabel("Hippocampal Volume (mL)")
axes[2].set_ylabel("Hippocampal MD (1e-3 mm2/s)")
axes[2].legend()
plt.tight_layout()
plt.show()


**생각해 보기:** `CU` 내부에서는 `copy_score_36`과 `recall_score_36`이 아밀로이드 양·음성 사이에 거의 차이가 없지만, 학습 전용 뇌지표인 `hippocampal_md`·`precuneus_md`·`amyloid_centiloid`는 뚜렷이 갈립니다. 이 정보를 `train` 학습(다중과제·교란 제거·표현 정렬)에 어떻게 녹여 내면 `test`의 ROCF 전용 추론에 도움이 될까요?

## 2. 정적 흑백 그림만 쓰는 스타터 기준선: 전체 AUROC 착시 확인

`starter_pipeline.py`는 `images/*.png`(정적 그림)만 읽어 예측합니다. 실행해 보면 전체(`CU + MCI`) AUROC는 약 `0.64–0.67`로 보이지만, `CU` 단독과 `MCI` 단독(진단군 내 층화 AUROC `cv_within_dx_auroc`)은 `0.50` 근처에 머무릅니다.


In [ ]:
out_starter = Path("outputs_notebook_starter")
summary_starter = run_static_starter(DATA_DIR, out_starter, requested_device="cpu", seed=42)
print(json.dumps(summary_starter, ensure_ascii=False, indent=2))


## 3. 정적 그림 vs 시간-색상·단계별 진행 패널(`render_process_panel`) 비교

`CU` 내부에서 36점 채점이 거의 만점인 `Aβ-` 참가자 1명과 `Aβ+` 참가자 1명을 골라, 정적 흑백 PNG와 5 Hz 궤적 기반 과정 패널을 나란히 비교합니다.


In [ ]:
cu_high = priv[(priv["diagnosis"] == "CU") & (priv["copy_score_36"] >= 35.0) & (priv["recall_score_36"] >= 34.0)]
sid_neg = cu_high[cu_high["amyloid_pos"] == 0]["subject_id"].iloc[0]
sid_pos = cu_high[cu_high["amyloid_pos"] == 1]["subject_id"].iloc[0]
print("비교 대상:", sid_neg, "(CU Ab-) vs", sid_pos, "(CU Ab+)")

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for r_idx, (sid, tag) in enumerate([(sid_neg, "CU Ab-"), (sid_pos, "CU Ab+")]):
    static_rec = Image.open(DATA_DIR / "images" / f"{sid}_recall.png")
    c_df = pd.read_csv(DATA_DIR / "strokes" / f"{sid}_copy.csv")
    r_df = pd.read_csv(DATA_DIR / "strokes" / f"{sid}_recall.csv")
    panel = rocf_synth.render_process_panel(c_df, r_df)

    axes[r_idx, 0].imshow(static_rec, cmap="gray")
    axes[r_idx, 0].set_title(f"{sid} ({tag}) - Static Recall PNG")
    axes[r_idx, 0].axis("off")

    axes[r_idx, 1].imshow(panel)
    axes[r_idx, 1].set_title(f"{sid} ({tag}) - 5 Hz Color-Time & Progress Panel")
    axes[r_idx, 1].axis("off")

plt.tight_layout()
plt.show()


### (선택) OpenRouter VLM 탐색 훅 안내

개발 단계에서는 위 `render_process_panel` 이미지를 소수 표본에 대해 OpenRouter VLM API로 보내 시각 패턴과 오류 사례를 탐색할 수 있습니다. 아래 셀은 기본값(`USE_VLM = False`)으로 오프라인 상태를 유지하며, **최종 제출 `run_pipeline.py` 본체는 API 키 없이 오프라인으로 단독 실행**되어야 합니다.

- 실행 전 터미널에서 `export OPENROUTER_API_KEY="sk-or-..."`처럼 환경변수로만 키를 설정하세요.
- **보안 경고**: 실제 API 키 문자열을 노트북 코드 셀에 직접 적거나, 출력 셀에 키·응답 흔적이 남은 채로 Git에 커밋·공유하지 마세요.


In [ ]:
# 절대 OPENROUTER_API_KEY 문자열을 코드에 직접 적지 말고, 터미널에서 export OPENROUTER_API_KEY="sk-or-..."로 설정하세요.
USE_VLM = False  # 시각 탐색이 필요할 때만 True로 바꾸고 환경변수를 설정하세요
api_key = os.environ.get("OPENROUTER_API_KEY", "").strip()
if USE_VLM and not api_key:
    print("경고: USE_VLM=True이지만 OPENROUTER_API_KEY 환경변수가 비어 있습니다. 코드에 키를 직접 적지 말고 터미널에서 export OPENROUTER_API_KEY=\"sk-or-...\" 후 다시 실행하세요.")
elif USE_VLM and api_key:
    import base64, io, urllib.request
    c_df = pd.read_csv(DATA_DIR / "strokes" / f"{sid_pos}_copy.csv")
    r_df = pd.read_csv(DATA_DIR / "strokes" / f"{sid_pos}_recall.csv")
    buf = io.BytesIO()
    rocf_synth.render_process_panel(c_df, r_df).save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    payload = {
        "model": "google/gemini-2.5-flash",
        "temperature": 0.0,
        "max_tokens": 200,
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": "Inspect Copy vs Delayed Recall stroke order (blue=early, red=late) and purple pause rings. Summarize structural-junction pauses and skeleton-first order in 3 bullet points."},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
        ]}],
    }
    req = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        print(json.loads(resp.read().decode("utf-8"))["choices"][0]["message"]["content"])
else:
    print("오프라인 기본 모드(USE_VLM=False): VLM API 호출을 건너뜁니다.")


## 4. 획 셔플 대조군(`shuffle_trial_strokes`) 불변성 확인

`shuffle_trial_strokes`로 5 Hz 획의 순서와 멈춤 시간을 섞어도 완성된 정적 흑백 PNG 픽셀은 **단 1픽셀도 바뀌지 않음**을 확인합니다.


In [ ]:
r_orig = pd.read_csv(DATA_DIR / "strokes" / f"{sid_pos}_recall.csv")
r_shuf = rocf_synth.shuffle_trial_strokes(r_orig, np.random.default_rng(7))
img_orig = np.asarray(rocf_synth.render_static_png(r_orig))
img_shuf = np.asarray(rocf_synth.render_static_png(r_shuf))
print("정적 PNG 픽셀 100% 동일 여부:", bool(np.array_equal(img_orig, img_shuf)))
print(f"원본 총 소요시간: {r_orig['t_sec'].iloc[-1]:.1f}s -> 셔플 후: {r_shuf['t_sec'].iloc[-1]:.1f}s")


## 5. 정답 파일(`ground_truth.json`)과 블라인드 테스트 비교

모델 설계와 반증 대조군(`--null-effect`, 획 셔플) 검증을 모두 마친 뒤, `ground_truth.json`의 `test_labels`·`test_diagnoses`로 블라인드 `test`(120명) 예측을 채점합니다.


In [ ]:
gt = json.loads((DATA_DIR / "ground_truth.json").read_text(encoding="utf-8"))
pred = pd.read_csv(out_starter / "predictions.csv")
y_te = np.array([gt["test_labels"][sid] for sid in pred["subject_id"]], dtype=int)
dx_te = np.array([gt["test_diagnoses"][sid] for sid in pred["subject_id"]], dtype=str)

test_metrics = rocf_synth.compute_within_dx_metrics(y_te, pred["y_prob"].to_numpy(float), dx_te)
print("정적 스타터 블라인드 test 성능:", test_metrics)
print("참고(ground_truth.json reference_metrics):", gt["reference_metrics"])


## 설명해 보세요

1. 왜 `CU + MCI` 전체 AUROC(`~0.65–0.70`)만 보면 정적 그림 모델도 아밀로이드를 예측하는 것처럼 착각하기 쉬운가요?
2. `train_privileged.csv`의 뇌지표(`hippocampal_md`, `precuneus_md`, `hippocampal_vol`, `amyloid_centiloid`)를 학습 때 활용하되 평가(`test`)에는 넣지 않을 때, 어떤 학습 전략(교란 잔차화·다중과제 회귀·표현 정렬·지식 증류)이 진단군 내 AUROC를 실제로 높여 주었나요?
3. 테스트 획 셔플 대조군과 `--null-effect` 영효과 코호트는 각각 어떤 종류의 가짜 성능을 걸러 내나요?

**다른 데이터로 반복 (과제 5) 및 규격 검증:**
```bash
python generate_synthetic_rocf.py --seed 7 --out data_seed7
python generate_synthetic_rocf.py --null-effect --out data_null
python smoke_test.py --pipeline run_pipeline.py --device cpu
```
